# Gradient Boosting — Clasificador (XGBoost & LightGBM)

**Repositorio:** `autodidact/both/gradient_boosting/`

---

## 1. Idea General e Intuición

**Gradient Boosting** es un método de ensamble que construye árboles de forma **secuencial**: cada árbol nuevo aprende a corregir los **errores residuales** del árbol anterior. A diferencia del Bagging (Random Forest, que construye árboles en paralelo), el Boosting es inherentemente iterativo.

**Intuición paso a paso:**
1. Inicializar la predicción con el valor constante óptimo (ej. log-odds del prior).
2. Calcular los **pseudo-residuos** (gradiente negativo de la pérdida).
3. Ajustar un árbol poco profundo a esos residuos.
4. Actualizar la predicción sumando una fracción (`learning_rate`) de ese árbol.
5. Repetir hasta `n_estimators` iteraciones.

**¿Por qué "Gradient"?** Cada árbol nuevo aproxima el **gradiente negativo** de la función de pérdida respecto a las predicciones actuales. Es descenso de gradiente en el espacio funcional.

**XGBoost y LightGBM** son implementaciones altamente optimizadas con:
- Regularización L1/L2 sobre los pesos de los árboles.
- Procesamiento paralelo y aproximaciones de histograma.
- Manejo nativo de valores faltantes.

## 2. Fundamento Matemático

### 2.1 Formulación General

El modelo en la iteración $m$ es:
$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x})$$

Donde:
- $F_{m-1}(\mathbf{x})$: modelo acumulado hasta la iteración anterior.
- $\eta$: **learning rate** (tasa de aprendizaje), $\eta \in (0, 1]$.
- $h_m(\mathbf{x})$: árbol ajustado al gradiente negativo en la iteración $m$.

### 2.2 Pseudo-Residuos (Gradiente Negativo)

$$r_{im} = -\left[\frac{\partial \mathcal{L}(y_i, F(\mathbf{x}_i))}{\partial F(\mathbf{x}_i)}\right]_{F=F_{m-1}}$$

Para **log-loss** (clasificación binaria):
$$r_{im} = y_i - \sigma(F_{m-1}(\mathbf{x}_i)) = y_i - \hat{p}_{m-1,i}$$

Son exactamente los residuos de la probabilidad predicha — el árbol aprende *cuánto* estaba equivocado el modelo anterior.

### 2.3 Objetivo de XGBoost (con Regularización)

XGBoost añade términos de regularización al árbol $h_m$:

$$\mathcal{L}^{(m)} = \sum_{i=1}^{n}\left[g_i f_m(\mathbf{x}_i) + \frac{1}{2}h_i f_m^2(\mathbf{x}_i)\right] + \Omega(f_m)$$

Donde:
- $g_i = \partial_{\hat{y}} \mathcal{L}(y_i, \hat{y}^{(m-1)})$: gradiente de primer orden.
- $h_i = \partial_{\hat{y}}^2 \mathcal{L}(y_i, \hat{y}^{(m-1)})$: gradiente de segundo orden (Hessiano).
- $\Omega(f_m) = \gamma T + \frac{1}{2}\lambda\|\mathbf{w}\|^2 + \alpha\|\mathbf{w}\|_1$: regularización sobre el árbol.

El **Hessiano** distingue a XGBoost de GBDT clásico y acelera la convergencia.

### 2.4 Peso Óptimo de Hoja (XGBoost)

Para una hoja $j$ que contiene instancias $\mathcal{I}_j$:

$$w_j^* = -\frac{\sum_{i \in \mathcal{I}_j} g_i}{\sum_{i \in \mathcal{I}_j} h_i + \lambda}$$

### 2.5 LightGBM: Diferencias Clave

| Característica | XGBoost | LightGBM |
|---|---|---|
| **Crecimiento del árbol** | Level-wise (por niveles) | Leaf-wise (por hoja de mayor ganancia) |
| **Histogramas** | Aproximados | Nativo y muy eficiente |
| **Velocidad** | Rápido | Más rápido en datasets grandes |
| **Memoria** | Media | Baja |
| **GOSS** | No | Sí (muestrea gradientes grandes) |
| **EFB** | No | Sí (agrupa features mutuamente excluyentes) |

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Tienes un dataset **tabular** de cualquier tamaño — es el **estado del arte en tabular**.
- Hay **valores faltantes** (manejo nativo en XGBoost/LightGBM).
- Necesitas el **mejor rendimiento predictivo** con features heterogéneas.
- Participas en competiciones de ML (es el modelo más ganador en Kaggle tabular).

### ❌ NO lo uses cuando:
- Tus datos son **imágenes, audio o texto** sin features ingeniadas (usar Deep Learning).
- Necesitas **interpretabilidad causal** directa.
- El dataset es **muy pequeño** (< 500 muestras) — riesgo de sobreajuste.

## 4. Métricas de Evaluación

Las mismas que cualquier clasificador, pero con énfasis en:

| Métrica | Notas para Gradient Boosting |
|---|---|
| **ROC-AUC** | Estándar de industria para GB en problemas binarios |
| **Log-loss** | GB optimiza directamente esta pérdida |
| **F1-Score** | Para clases desbalanceadas |
| **Early stopping score** | Métrica usada en el set de validación para parar temprano |

## 5. Problemas y Casos Raros

### 5.1 Sobreajuste con `learning_rate` alto + muchos árboles
Con $\eta = 0.3$ y 1000 árboles el modelo memorizará.
**Solución:** Reducir $\eta$ (ej. 0.01–0.05) y aumentar `n_estimators` + **Early Stopping**.

### 5.2 Leaf-wise en LightGBM (riesgo de sobreajuste)
El crecimiento leaf-wise puede crear hojas muy profundas en una sola rama.
**Solución:** Controlar `num_leaves` (max 2^max_depth), `min_child_samples`.

### 5.3 Clases Desbalanceadas
**XGBoost:** `scale_pos_weight = n_neg / n_pos`.
**LightGBM:** `is_unbalance=True` o `class_weight='balanced'`.

### 5.4 Fuga de Datos (Data Leakage) en Early Stopping
El set de validación para early stopping **no debe** ser el mismo que el de CV.
**Solución:** Usar un split dedicado o `eval_set` con datos nunca vistos.

## 6. Hiperparámetros Clave

| Parámetro | XGBoost | LightGBM | Efecto |
|---|---|---|---|
| Árboles | `n_estimators` | `n_estimators` | Más → mejor pero más lento |
| Tasa aprendizaje | `learning_rate` | `learning_rate` | ↓ → más robusto, necesita más árboles |
| Profundidad | `max_depth` | `max_depth` | XGB: 3-8; LGBM: usar `num_leaves` |
| Hojas | — | `num_leaves` | Max hojas del árbol (LGBM leaf-wise) |
| Submuestreo | `subsample` | `bagging_fraction` | Fracción de muestras por árbol |
| Features | `colsample_bytree` | `feature_fraction` | Fracción de features por árbol |
| Reg. L2 | `reg_lambda` | `lambda_l2` | Penaliza pesos de hojas |
| Reg. L1 | `reg_alpha` | `lambda_l1` | Sparsity en pesos de hojas |
| Min muestras hoja | `min_child_weight` | `min_child_samples` | Regularización |

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay, log_loss
)
from sklearn.inspection import permutation_importance

import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
print(f'XGBoost  v{xgb.__version__}')
print(f'LightGBM v{lgb.__version__}')

---
## 7. Clasificación — Breast Cancer

In [ ]:
# =============================================================================
# CARGA Y SPLIT
# GB no requiere escalado, pero lo dejamos disponible
# =============================================================================
bc = load_breast_cancer(as_frame=True)
X, y = bc.data, bc.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
# Validación interna para early stopping (separada del test)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)
print(f'Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

In [ ]:
# =============================================================================
# XGBOOST — Clasificador con Early Stopping
# early_stopping_rounds: para el entrenamiento si la métrica de val
# no mejora en N rondas consecutivas → evita sobreajuste automáticamente
# =============================================================================
xgb_clf = xgb.XGBClassifier(
    n_estimators=1000,          # Máximo de árboles (early stopping parará antes)
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,             # Regularización L2
    reg_alpha=0.1,              # Regularización L1
    scale_pos_weight=1,         # Para desbalanceo: n_neg/n_pos
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=SEED,
    n_jobs=-1
)

xgb_clf.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

y_pred_xgb  = xgb_clf.predict(X_test)
y_proba_xgb = xgb_clf.predict_proba(X_test)[:, 1]

print(f'XGBoost:')
print(f'  Árboles usados : {xgb_clf.best_iteration + 1}')
print(f'  Accuracy       : {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'  ROC-AUC        : {roc_auc_score(y_test, y_proba_xgb):.4f}')
print(f'  Log-loss       : {log_loss(y_test, y_proba_xgb):.4f}')

In [ ]:
# =============================================================================
# LIGHTGBM — Clasificador con Early Stopping
# LightGBM usa callbacks para early stopping (API distinta a XGBoost)
# =============================================================================
lgb_clf = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,              # Parámetro clave en LGBM (leaf-wise growth)
    max_depth=-1,               # -1 = sin límite (controlado por num_leaves)
    subsample=0.8,
    feature_fraction=0.8,
    lambda_l2=1.0,
    lambda_l1=0.1,
    min_child_samples=20,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

callbacks = [lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
lgb_clf.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=callbacks
)

y_pred_lgb  = lgb_clf.predict(X_test)
y_proba_lgb = lgb_clf.predict_proba(X_test)[:, 1]

print(f'LightGBM:')
print(f'  Árboles usados : {lgb_clf.best_iteration_}')
print(f'  Accuracy       : {accuracy_score(y_test, y_pred_lgb):.4f}')
print(f'  ROC-AUC        : {roc_auc_score(y_test, y_proba_lgb):.4f}')
print(f'  Log-loss       : {log_loss(y_test, y_proba_lgb):.4f}')

In [ ]:
# =============================================================================
# VISUALIZACIÓN 1: Curva de aprendizaje (training loss vs iteración)
# Muestra cuándo el early stopping activó y cómo evoluciona la pérdida
# =============================================================================
xgb_val_loss = xgb_clf.evals_result()['validation_0']['logloss']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(xgb_val_loss, color='steelblue', lw=2, label='XGBoost — Val Log-loss')
ax.axvline(xgb_clf.best_iteration, color='steelblue', linestyle='--', alpha=0.7,
           label=f'Early stop: iter {xgb_clf.best_iteration+1}')
ax.set_xlabel('Número de árboles', fontsize=12)
ax.set_ylabel('Log-loss (validación)', fontsize=12)
ax.set_title('Curva de Aprendizaje — Early Stopping', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 2: Importancia de Features — XGBoost (4 tipos)
# weight: frecuencia de uso | gain: mejora promedio | cover: muestras cubiertas
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

importance_types = [('weight', 'Frecuencia de uso'), ('gain', 'Ganancia promedio')]
for ax, (imp_type, title) in zip(axes, importance_types):
    imp = pd.Series(xgb_clf.get_booster().get_score(importance_type=imp_type))
    imp = imp.sort_values(ascending=False).head(15)
    ax.barh(imp.index[::-1], imp.values[::-1], color='steelblue', alpha=0.8)
    ax.set_title(f'XGBoost Importancia ({title})', fontsize=12, fontweight='bold')
    ax.set_xlabel(imp_type)

plt.suptitle('Importancia de Features — XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 3: ROC Curves + Confusion Matrices (XGB vs LGBM)
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ROC
RocCurveDisplay.from_predictions(y_test, y_proba_xgb, ax=axes[0], name='XGBoost')
RocCurveDisplay.from_predictions(y_test, y_proba_lgb, ax=axes[0], name='LightGBM')
axes[0].plot([0,1],[0,1],'k--', label='Azar')
axes[0].set_title('Curvas ROC — XGBoost vs LightGBM', fontweight='bold')
axes[0].legend()

# CM XGBoost
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_xgb),
                       display_labels=bc.target_names).plot(
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Conf. Matrix — XGBoost', fontweight='bold')

# CM LightGBM
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lgb),
                       display_labels=bc.target_names).plot(
    ax=axes[2], colorbar=False, cmap='Greens')
axes[2].set_title('Conf. Matrix — LightGBM', fontweight='bold')

plt.suptitle('Evaluación Comparativa — Clasificador Breast Cancer',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# EFECTO DEL LEARNING RATE — trade-off velocidad vs calidad
# =============================================================================
lr_values = [0.001, 0.01, 0.05, 0.1, 0.3]
results_lr = {}

for lr in lr_values:
    clf_lr = xgb.XGBClassifier(
        n_estimators=500, learning_rate=lr, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', early_stopping_rounds=20,
        random_state=SEED, n_jobs=-1, verbosity=0
    )
    clf_lr.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    y_p = clf_lr.predict_proba(X_test)[:,1]
    results_lr[lr] = {
        'roc_auc': roc_auc_score(y_test, y_p),
        'trees': clf_lr.best_iteration + 1
    }

print(f'{"LR":>6}  {"ROC-AUC":>8}  {"Árboles":>8}')
print('-' * 30)
for lr, res in results_lr.items():
    print(f'{lr:>6.3f}  {res["roc_auc"]:>8.4f}  {res["trees"]:>8}')

In [ ]:
# =============================================================================
# COMPARATIVA FINAL: sklearn GBM vs XGBoost vs LightGBM
# =============================================================================
from sklearn.ensemble import GradientBoostingClassifier

models_compare = {
    'sklearn GBM':  GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                max_depth=4, random_state=SEED),
    'XGBoost':      xgb.XGBClassifier(n_estimators=200, learning_rate=0.05,
                                       max_depth=4, random_state=SEED, verbosity=0, n_jobs=-1),
    'LightGBM':     lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                        num_leaves=15, random_state=SEED, n_jobs=-1, verbose=-1)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f'{"Modelo":<16}  {"ROC-AUC":>8}  {"Log-loss":>9}  {"Accuracy":>9}')
print('-' * 50)
for name, model in models_compare.items():
    roc  = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
    lls  = -cross_val_score(model, X, y, cv=skf, scoring='neg_log_loss')
    acc  = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    print(f'{name:<16}  {roc.mean():.4f}±{roc.std():.3f}  '
          f'{lls.mean():.4f}±{lls.std():.3f}  {acc.mean():.4f}±{acc.std():.3f}')

---
## 8. Resumen Clasificador

| Aspecto | XGBoost | LightGBM |
|---|---|---|
| **Crecimiento árbol** | Level-wise | Leaf-wise |
| **Velocidad** | Rápido | Más rápido en datos grandes |
| **Parámetro clave** | `max_depth` | `num_leaves` |
| **Early stopping** | `early_stopping_rounds` | `lgb.early_stopping()` callback |
| **Desbalanceo** | `scale_pos_weight` | `is_unbalance=True` |

### ✅ Checklist:
- [ ] ¿Usé early stopping con un set de validación dedicado?
- [ ] ¿Probé `learning_rate` bajo (0.01-0.05) con muchos árboles?
- [ ] ¿Comparé importancia XGBoost `gain` vs permutation importance?
- [ ] ¿Usé `scale_pos_weight` / `is_unbalance` si hay desbalanceo?

---
*Ver notebook de regresión: `gb_regressor.ipynb`*